# TRUJAMAn 🧙 · 1. Training

**T**extual **R**ecognition **U**nit for **J**udeo-**A**rabic (and Hebrew) in **M**edieval **A**nthologies · *beta v0.1.0*

Running the cell below trains the word-level Judeo-Arabic / Hebrew classifier and evaluates it:

1. reads every work in `1.Data/b.Training/b.1.Training/` and trains on it,
2. saves the model to `trujaman_v0.1-beta.joblib` (repository root),
3. scores every work in `1.Data/b.Training/b.2.Evaluation/` and writes `evaluation_scores.csv` there.

**The texts are not in this repository.** They come from the Friedberg Jewish Manuscript Society (FJMS) Judeo-Arabic portal, whose terms of use do not allow redistribution. The works used are listed in `works.csv` in each folder and in `1.Data/a.Sources/sources.csv`. With your own FJMS access, export each work as a CSV with the columns `wID` (page.line.index), `word` and `lang` (`JA`, `HE`; other labels such as `header` are ignored), name it after its `codeName`, and place it in the matching folder.

The split keeps every work by **Ben Hofni** out of training, so the evaluation includes an author the model has never seen.

In [1]:
"""
Judeo-Arabic / Hebrew word-level language ID.

Trains on every CSV in TRAIN_DIR, evaluates on every CSV in EVAL_DIR,
saves the model to MODEL_PATH.

Each CSV needs: wID (page.line.index), word, lang
Rows labelled anything other than JA / HE (e.g. 'header') are dropped.

All diacritics are deleted before featurisation, so the classifier cannot
depend on how consistently a manuscript is pointed, and works using
different conventions (upper dot, apostrophe, geresh) can be pooled.
"""
# ===========================================================================
TRAIN_DIR   = "1.Data/b.Training/b.1.Training"     # folder of CSVs to train on
EVAL_DIR    = "1.Data/b.Training/b.2.Evaluation"   # folder of CSVs to evaluate on
MODEL_PATH  = "trujaman_v0.1-beta.joblib"
SCORES_PATH = "1.Data/b.Training/b.2.Evaluation/evaluation_scores.csv"
PRED_PATH   = "eval_predictions.csv"               # contains the texts: never published
# ===========================================================================

import re, glob, os
import numpy as np, pandas as pd, joblib
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, roc_auc_score)

# 05C4 upper dot | 05BF rafe | 05F3/05F4 geresh, gershayim | ' " | punctuation
# The app (streamlit_app.py) uses the same pattern: change both or neither.
STRIP = re.compile("[\u05c4\u05bf\u05f3\u05f4\u0027\u0022,.;:()\\[\\]\\-\u2013\u2014?!\u2026]")


def normalise(s):
    return STRIP.sub("", str(s))


def load_dir(folder):
    """Read every work CSV in a folder into one frame, tagged by source file."""
    listed = pd.read_csv(os.path.join(folder, "works.csv")).codeName
    paths = sorted(p for p in glob.glob(os.path.join(folder, "*.csv"))
                   if os.path.basename(p) not in ("works.csv", "evaluation_scores.csv"))
    if not paths:
        raise FileNotFoundError(
            f"No texts found in {folder}. The FJMS texts are not distributed with "
            "this repository: see the note above on how to add them.")
    found = {os.path.splitext(os.path.basename(p))[0] for p in paths}
    missing_works = sorted(set(listed) - found)
    if missing_works:
        print(f"WARNING: {len(missing_works)} work(s) listed in works.csv are missing "
              f"from {folder}: {missing_works}")
    frames = []
    for p in paths:
        d = pd.read_csv(p)
        missing = {"wID", "word", "lang"} - set(d.columns)
        if missing:
            raise ValueError(f"{p} is missing column(s): {sorted(missing)}")
        d = d[d.lang.isin(["JA", "HE"])].copy()
        d["work"] = os.path.splitext(os.path.basename(p))[0]
        d["doc"] = d.work + ":" + d.wID.astype(str).str.split(".").str[0]
        d["w"] = d.word.map(normalise)
        frames.append(d[d.w.str.len() > 0])
    return pd.concat(frames, ignore_index=True)


def features(d):
    """Target word marked with ~, flanked by its neighbours on the same page."""
    prev = d.w.groupby(d.doc).shift(1).fillna("")
    nxt = d.w.groupby(d.doc).shift(-1).fillna("")
    return (prev + " ~" + d.w + "~ " + nxt).values


def build():
    return make_pipeline(
        TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 5),
                        min_df=2, sublinear_tf=True),
        LogisticRegression(max_iter=3000, C=4, class_weight="balanced"))


def report(name, y, score):
    pred = (score > 0).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    print(f"\n--- {name}  ({len(y):,} words, {y.mean():.1%} Hebrew) ---")
    print(f"AUC {roc_auc_score(y, score):.4f}   "
          f"JA false-positive rate {fp / max(fp + tn, 1):.3f}")
    print(classification_report(y, pred, labels=[0, 1], target_names=["JA", "HE"],
                                digits=3, zero_division=0))
    return {"work": name, "words": len(y), "pct_HE": round(y.mean(), 3),
            "accuracy": round(accuracy_score(y, pred), 4),
            "AUC": round(roc_auc_score(y, score), 4),
            "F1_JA": round(f1_score(y, pred, pos_label=0, zero_division=0), 4),
            "F1_HE": round(f1_score(y, pred, pos_label=1, zero_division=0), 4)}


# ---------------------------------------------------------------- train ----
train = load_dir(TRAIN_DIR)
print("TRAINING DATA")
print(train.groupby("work").agg(words=("w", "size"),
                                pct_HE=("lang", lambda x: (x == "HE").mean())).to_string())
print(f"total: {len(train):,} words")

clf = build().fit(features(train), (train.lang == "HE").astype(int).values)

os.makedirs(os.path.dirname(MODEL_PATH) or ".", exist_ok=True)
joblib.dump(clf, MODEL_PATH)
print(f"\nmodel saved to {MODEL_PATH}")

# ----------------------------------------------------------- evaluate ----
ev = load_dir(EVAL_DIR)
overlap = set(ev.work) & set(train.work)
if overlap:
    print(f"\nWARNING: {sorted(overlap)} appear in both folders - "
          "these scores are not held-out.")

print("\n" + "=" * 60 + "\nEVALUATION\n" + "=" * 60)
y_all = (ev.lang == "HE").astype(int).values
s_all = clf.decision_function(features(ev))

scores = []
for w in ev.work.unique():                       # one report per work
    m = (ev.work == w).values
    scores.append(report(w, y_all[m], s_all[m]))

if ev.work.nunique() > 1:
    scores.append(report("ALL EVALUATION WORKS POOLED", y_all, s_all))

pd.DataFrame(scores).to_csv(SCORES_PATH, index=False)
print(f"\nscores per work written to {SCORES_PATH}")

ev["p_HE"] = clf.predict_proba(features(ev))[:, 1]
ev["pred"] = np.where(ev.p_HE > 0.5, "HE", "JA")
ev[["work", "wID", "word", "lang", "pred", "p_HE"]].to_csv(PRED_PATH, index=False)
print(f"per-word predictions written to {PRED_PATH} (local only, not published)")

TRAINING DATA
                        words    pct_HE
work                                   
AbMaimonides_brASHyt    78859  0.257345
AbMaimonides_hmspyk1    54240  0.264786
AbMaimonides_hmspyk2     8137  0.282045
AbMaimonides_hmspyk3    38495  0.253903
AbrBenSol_SHmvAl1       62948  0.665994
AbrBenSol_SHmvAl2       49773  0.663914
AbrBenSol_SHvpTym       33123  0.624370
AbrBenSol_mlKHym        71511  0.565214
DavidBenAbraMaim_prky   57658  0.209216
IbnBalam_SHlSHh          7046  0.447772
IbnBalam_SHvpTym         4154  0.319451
IbnBalam_try            11454  0.353239
IbnBalam_yHzkAl         10174  0.305091
IbnBalam_ySH`yhv        20963  0.302151
IbnBalam_yrmyhv          8371  0.333413
IbnJanah_spr1            4510  0.132816
IbnJanah_spr2          107571  0.382352
Maim_Agrvt              32279  0.206047
Maim_mvrh              142518  0.121213
Maim_prvSH1             88805  0.289150
Maim_prvSH2             74121  0.319680
Maim_prvSH3             55626  0.356272
Maim_prvSH4             96


model saved to trujaman_v0.1-beta.joblib



EVALUATION



--- AbMaimonides_tSHvbvt  (37,745 words, 48.1% Hebrew) ---
AUC 0.9959   JA false-positive rate 0.027
              precision    recall  f1-score   support

          JA      0.973     0.973     0.973     19576
          HE      0.971     0.971     0.971     18169

    accuracy                          0.972     37745
   macro avg      0.972     0.972     0.972     37745
weighted avg      0.972     0.972     0.972     37745


--- AbrBenSol_yhvSH`  (23,262 words, 74.8% Hebrew) ---
AUC 0.9967   JA false-positive rate 0.037
              precision    recall  f1-score   support

          JA      0.955     0.963     0.959      5864
          HE      0.987     0.985     0.986     17398

    accuracy                          0.979     23262
   macro avg      0.971     0.974     0.972     23262
weighted avg      0.979     0.979     0.979     23262


--- BenHofniG_`SHr  (9,794 words, 7.1% Hebrew) ---
AUC 0.9954   JA false-positive rate 0.014
              precision    recall  f1-score   suppor


--- HaiG_mSHpTy  (22,892 words, 39.8% Hebrew) ---
AUC 0.9936   JA false-positive rate 0.026
              precision    recall  f1-score   support

          JA      0.967     0.974     0.970     13782
          HE      0.961     0.949     0.955      9110

    accuracy                          0.964     22892
   macro avg      0.964     0.962     0.963     22892
weighted avg      0.964     0.964     0.964     22892


--- Hayyuj_spr  (17,356 words, 40.1% Hebrew) ---
AUC 0.9805   JA false-positive rate 0.041
              precision    recall  f1-score   support

          JA      0.933     0.959     0.946     10397
          HE      0.936     0.897     0.916      6959

    accuracy                          0.934     17356
   macro avg      0.934     0.928     0.931     17356
weighted avg      0.934     0.934     0.934     17356


--- HefBenYatz_spr  (18,468 words, 57.3% Hebrew) ---
AUC 0.9952   JA false-positive rate 0.050
              precision    recall  f1-score   support

          


--- IbnPaquda_tvrt  (88,575 words, 14.7% Hebrew) ---
AUC 0.9982   JA false-positive rate 0.019
              precision    recall  f1-score   support

          JA      0.996     0.981     0.989     75566
          HE      0.901     0.979     0.938     13009

    accuracy                          0.981     88575
   macro avg      0.949     0.980     0.964     88575
weighted avg      0.982     0.981     0.981     88575


--- IbnQuraysh_Aygrt  (18,034 words, 45.5% Hebrew) ---
AUC 0.9582   JA false-positive rate 0.127
              precision    recall  f1-score   support

          JA      0.929     0.873     0.900      9821
          HE      0.858     0.920     0.888      8213

    accuracy                          0.894     18034
   macro avg      0.893     0.896     0.894     18034
weighted avg      0.896     0.894     0.894     18034


--- JosBenSimeon_Aygrt  (10,064 words, 13.4% Hebrew) ---
AUC 0.9976   JA false-positive rate 0.015
              precision    recall  f1-score   suppor


--- NethBenIsa_mAvr  (146,168 words, 66.8% Hebrew) ---
AUC 0.9972   JA false-positive rate 0.037
              precision    recall  f1-score   support

          JA      0.970     0.963     0.967     48599
          HE      0.982     0.985     0.983     97569

    accuracy                          0.978    146168
   macro avg      0.976     0.974     0.975    146168
weighted avg      0.978     0.978     0.978    146168


--- NissimG_HmSHh  (69,228 words, 50.9% Hebrew) ---
AUC 0.9922   JA false-positive rate 0.045
              precision    recall  f1-score   support

          JA      0.969     0.955     0.962     33959
          HE      0.957     0.971     0.964     35269

    accuracy                          0.963     69228
   macro avg      0.963     0.963     0.963     69228
weighted avg      0.963     0.963     0.963     69228


--- ObasiahMaim_mAmr  (8,321 words, 25.5% Hebrew) ---
AUC 0.9955   JA false-positive rate 0.032
              precision    recall  f1-score   support

 

              precision    recall  f1-score   support

          JA      0.994     0.970     0.982     22267
          HE      0.890     0.976     0.931      5474

    accuracy                          0.971     27741
   macro avg      0.942     0.973     0.956     27741
weighted avg      0.973     0.971     0.972     27741


--- alHarizi_ms`y  (3,339 words, 64.7% Hebrew) ---
AUC 0.9796   JA false-positive rate 0.123
              precision    recall  f1-score   support

          JA      0.932     0.877     0.904      1179
          HE      0.935     0.965     0.950      2160

    accuracy                          0.934      3339
   macro avg      0.934     0.921     0.927      3339
weighted avg      0.934     0.934     0.934      3339


--- alMuqammis_`SHrym  (26,428 words, 0.4% Hebrew) ---
AUC 0.9728   JA false-positive rate 0.017
              precision    recall  f1-score   support

          JA      0.999     0.983     0.991     26329
          HE      0.139     0.747     0.235  

AUC 0.9936   JA false-positive rate 0.036
              precision    recall  f1-score   support

          JA      0.980     0.964     0.972    683038
          HE      0.938     0.965     0.951    387879

    accuracy                          0.964   1070917
   macro avg      0.959     0.964     0.962   1070917
weighted avg      0.965     0.964     0.964   1070917




scores per work written to 1.Data/b.Training/b.2.Evaluation/evaluation_scores.csv


per-word predictions written to eval_predictions.csv (local only, not published)
